In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score
from sklearn.metrics import classification_report

In [3]:
# Load predictions
predictions_df = pd.read_csv(snakemake.input.predictions)
print(f"Loaded predictions with shape: {predictions_df.shape}")
print(f"Columns: {list(predictions_df.columns)}")

Loaded predictions with shape: (2475, 8)
Columns: ['Unnamed: 0', 'score_for_normal cells', 'score_for_tumor cells', 'score_for_infiltrating cells', 'score_for_tertiary lymphoid structures', 'predicted_labels', 'label', 'is_correct']


In [12]:
# Extract ground truth and predictions

y_true = predictions_df['label']
y_pred = predictions_df['predicted_labels']

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted')
f1_macro = f1_score(y_true, y_pred, average='macro')

print(f"Accuracy: {accuracy:.3f}")
print(f"F1-score (weighted): {f1:.3f}")

print(f"F1-score (macro): {f1_macro:.3f}")

# Generate confusion matrix
cm = confusion_matrix(y_true, y_pred)
unique_labels = sorted(list(set(y_true) | set(y_pred)))

print(f"\nClassification Report:")
print(classification_report(y_true, y_pred))


Accuracy: 0.088
F1-score (weighted): 0.056
F1-score (macro): 0.214

Classification Report:
              precision    recall  f1-score   support

        INFL       0.08      0.87      0.15       112
         NOR       0.04      0.00      0.00      1646
         TLS       0.72      0.61      0.66       138
         TUM       0.03      0.06      0.04       579

    accuracy                           0.09      2475
   macro avg       0.22      0.38      0.21      2475
weighted avg       0.08      0.09      0.06      2475



In [9]:
# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=unique_labels, yticklabels=unique_labels)
plt.title(f'Confusion Matrix - {snakemake.wildcards.dataset}\n{snakemake.wildcards.metadata_col}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()

# Save plot
Path(snakemake.output.confusion_matrix).parent.mkdir(parents=True, exist_ok=True)
plt.savefig(snakemake.output.confusion_matrix, dpi=300, bbox_inches='tight')
plt.close()

print(f"Saved confusion matrix to {snakemake.output.confusion_matrix}")

Saved confusion matrix to /home/groups/zinaida/moritzs/cellwhisperer_private/results/plots/spotwhisperer_evaluation/spotwhisperer_cellxgene_census__archs4_geo__hest1k__quilt1m/plots/lung_tissue/confusion_matrix_region_type_expert_annotation_by_cell.png


In [13]:
# Save performance metrics
metrics = {
    'dataset': snakemake.wildcards.dataset,
    'metadata_col': snakemake.wildcards.metadata_col,
    'grouping': snakemake.wildcards.grouping,
    'accuracy': float(accuracy),
    'f1_weighted': float(f1),
    'f1_macro': float(f1_macro),
    'n_samples': len(predictions_df),
    'n_classes': len(unique_labels)
}

Path(snakemake.output.performance_metrics).parent.mkdir(parents=True, exist_ok=True)
with open(snakemake.output.performance_metrics, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"Saved performance metrics to {snakemake.output.performance_metrics}")
print(f"Metrics: {metrics}")

Saved performance metrics to /home/groups/zinaida/moritzs/cellwhisperer_private/results/plots/spotwhisperer_evaluation/spotwhisperer_cellxgene_census__archs4_geo__hest1k__quilt1m/performance/lung_tissue/region_type_expert_annotation_by_cell_metrics.json
Metrics: {'dataset': 'lung_tissue', 'metadata_col': 'region_type_expert_annotation', 'grouping': 'by_cell', 'accuracy': 0.08767676767676767, 'f1_weighted': 0.05578122079241292, 'f1_macro': 0.21373025095122175, 'n_samples': 2475, 'n_classes': 4}
